# 00e — POI & Road Network Download

## Overview

This notebook downloads two static spatial datasets for each city from
**OpenStreetMap** via `osmnx`:

1. **Points of Interest (POI)** — traffic-generating amenities, shops and land uses
   saved as a single GeoPackage with a `category` column identifying each POI type.
2. **Road network** — driveable roads only, saved as separate node and edge
   GeoPackages for later grid-level feature derivation (road length per cell,
   intersection density, distance-to-road, etc.)

Both datasets are bounded by the **clipped land-only city boundary** produced by
notebook 00a, consistent with the spatial extent used throughout the pipeline.

---

## POI Categories

Categories are chosen to maximise relevance for predicting the **Traffic Congestion
Index (TCI)**. Each is a known traffic generator or attractor:

| OSM key | OSM values | Traffic rationale |
|---|---|---|
| `amenity` | `fuel` | Direct traffic generator — vehicles must stop |
| `amenity` | `parking` | Direct traffic attractor — destination for vehicles |
| `amenity` | `bus_station` | Public transport interchange — pedestrian/vehicle flows |
| `amenity` | `school` | Strong peak-hour congestion generator |
| `amenity` | `hospital` | All-day traffic attractor |
| `amenity` | `restaurant`, `fast_food` | Short-stop vehicle trips |
| `shop` | `supermarket`, `mall` | High-volume vehicle destinations |
| `shop` | `car` | Car dealership — vehicle-oriented land use |
| `landuse` | `industrial` | Freight traffic generator |
| `landuse` | `commercial`, `retail` | Dense vehicle trip attractors |
| `public_transport` | `station` | Rail/metro — interchange congestion |

Each category is saved as a separate row in the output GeoPackage with a `category`
column, so grid-level aggregation in a later notebook can produce one column per
category.

---

## Road Network

Downloaded as a **driveable road network** (`network_type='drive'`). Saved as:
- **Edges** — road segments with geometry, length, road type (`highway` tag)
- **Nodes** — intersections and endpoints

Both are needed for later derivation of features such as road length per grid cell,
intersection density, and minimum distance from each cell centroid to the nearest road.

---

## Workflow

```
For each city
    │
    ├── Load clipped boundary from 00a
    │    └── data/raw/{city}/city_boundary/{city}_boundary_clipped.gpkg
    │
    ├── Download POI (if not already on disk)
    │    ├── Query OSM for each category within the boundary polygon
    │    ├── Tag each feature with its category
    │    ├── Combine all categories into one GeoDataFrame
    │    └── Save → data/raw/{city}/poi/{city}_poi.gpkg
    │
    └── Download road network (if not already on disk)
         ├── Download driveable graph within the boundary polygon
         ├── Extract nodes and edges GeoDataFrames
         ├── Save edges → data/raw/{city}/road_network/{city}_roads_edges.gpkg
         └── Save nodes → data/raw/{city}/road_network/{city}_roads_nodes.gpkg
```

---

## Output Structure

```
data/raw/
└── {city}/
    ├── poi/
    │   └── {city}_poi.gpkg              ← all POI with 'category' column
    └── road_network/
        ├── {city}_roads_edges.gpkg      ← road segments (geometry, length, highway type)
        └── {city}_roads_nodes.gpkg      ← intersections and endpoints
```

---

## Prerequisites

- Notebook 00a must have run for all cities so that the clipped boundary files
  exist at `data/raw/{city}/city_boundary/{city}_boundary_clipped.gpkg`.
- Internet connection required (OSM downloads). Re-running is safe — existing
  files are skipped automatically.

## 1. Setup & Configuration

In [ ]:
# !pip install osmnx geopandas shapely pandas -q

In [ ]:
from pathlib import Path
import sys
import warnings

import osmnx as ox
import geopandas as gpd
import pandas as pd

warnings.filterwarnings("ignore")

# ── Project root & src path ───────────────────────────────────────────────────
current_path = Path.cwd().resolve()
PROJECT_ROOT = current_path.parent if current_path.name == "notebooks" else current_path

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Data directory ────────────────────────────────────────────────────────────
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── OSMnx settings ────────────────────────────────────────────────────────────
ox.settings.timeout   = 300   # seconds before an OSM request times out
ox.settings.use_cache = True  # cache OSM responses — repeated runs are fast
ox.settings.overpass_endpoint = "https://overpass.kumi.systems/api/interpreter"

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data dir : {RAW_DIR}")

## 2. City Configuration

Edit `CITIES` to match the cities you want to process. The list must be consistent
with the city slugs used in notebooks 00a–00d so that the clipped boundary files
can be resolved correctly.

In [ ]:
# ── City list ─────────────────────────────────────────────────────────────────
# Slugs must match the directory names under data/raw/ created by notebook 00a.
# Must be consistent with CITIES in notebooks 00a–00d.
CITIES = [
    "abuja",
    "algiers",
    "baghdad",
    "buenos_aires",
    "cairo",
    "cape_town",
    "dakar",
    "lima",
    "los_angeles",
    "madrid",
    "mexico_city",
    "mumbai",
    "new_york",
    "santiago",
    "yangon",
]

# ── POI category definitions ───────────────────────────────────────────────────
# Each entry maps a human-readable category name to its OSM key-value pair.
# The category name becomes the value in the 'category' column of the output
# GeoPackage, allowing one column per category in later grid aggregation.
#
# Categories are chosen for TCI prediction relevance:
#   - Direct vehicle generators: fuel, parking, car dealerships
#   - Peak-hour attractors: schools, hospitals
#   - Commercial attractors: restaurants, fast food, supermarkets, malls
#   - Land use generators: industrial, commercial, retail
#   - Interchange points: bus stations, rail/metro stations
POI_CATEGORIES = {
    # ── Amenities ─────────────────────────────────────────────────────────────
    "fuel"        : {"amenity": "fuel"},
    "parking"     : {"amenity": "parking"},
    "bus_station" : {"amenity": "bus_station"},
    "school"      : {"amenity": "school"},
    "hospital"    : {"amenity": "hospital"},
    "restaurant"  : {"amenity": "restaurant"},
    "fast_food"   : {"amenity": "fast_food"},
    # ── Shops ─────────────────────────────────────────────────────────────────
    "supermarket" : {"shop": "supermarket"},
    "mall"        : {"shop": "mall"},
    "car_dealer"  : {"shop": "car"},
    # ── Land use ──────────────────────────────────────────────────────────────
    "industrial"  : {"landuse": "industrial"},
    "commercial"  : {"landuse": "commercial"},
    "retail"      : {"landuse": "retail"},
    # ── Public transport ──────────────────────────────────────────────────────
    "pt_station"  : {"public_transport": "station"},
}

print(f"Cities        : {len(CITIES)}")
print(f"POI categories: {len(POI_CATEGORIES)}  → {list(POI_CATEGORIES.keys())}")

## 3. Helper Functions

In [ ]:
# ── Path helpers ──────────────────────────────────────────────────────────────

import time


def clipped_boundary_path(city: str) -> Path:
    """Return path to the clipped boundary GeoPackage produced by notebook 00a."""
    return RAW_DIR / city / "city_boundary" / f"{city}_boundary_clipped.gpkg"

def poi_dir(city: str) -> Path:
    """Return (and create) the POI output directory for a city."""
    p = RAW_DIR / city / "poi"
    p.mkdir(parents=True, exist_ok=True)
    return p

def road_dir(city: str) -> Path:
    """Return (and create) the road network output directory for a city."""
    p = RAW_DIR / city / "road_network"
    p.mkdir(parents=True, exist_ok=True)
    return p

def poi_output_path(city: str) -> Path:
    return poi_dir(city) / f"{city}_poi.gpkg"

def edges_output_path(city: str) -> Path:
    return road_dir(city) / f"{city}_roads_edges.gpkg"

def nodes_output_path(city: str) -> Path:
    return road_dir(city) / f"{city}_roads_nodes.gpkg"


# ── Boundary loader ────────────────────────────────────────────────────────────

def load_boundary(city: str) -> gpd.GeoDataFrame:
    """
    Load the clipped land-only city boundary produced by notebook 00a.
    Raises FileNotFoundError with a clear message if missing.
    """
    path = clipped_boundary_path(city)
    if not path.exists():
        raise FileNotFoundError(
            f"Clipped boundary not found for '{city}'.\n"
            f"  Expected: {path}\n"
            f"  Run notebook 00a first."
        )
    return gpd.read_file(path).to_crs("EPSG:4326")


# ── POI download ───────────────────────────────────────────────────────────────

import time
import random

# ── Overpass mirror endpoints — tried in order on failure ─────────────────
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

def _set_overpass_endpoint(endpoint: str) -> None:
    """Point osmnx to a specific Overpass endpoint."""
    ox.settings.overpass_url = endpoint
    ox.settings.nominatim_url = "https://nominatim.openstreetmap.org/"

def _with_retry(fn, *args, max_retries: int = 5, base_wait: float = 10.0, **kwargs):
    """
    Call fn(*args, **kwargs) with exponential backoff + Overpass endpoint rotation.
    Waits base_wait * 2^attempt seconds between retries, plus random jitter.
    """
    last_exc = None
    for attempt in range(max_retries):
        # Rotate endpoint on each retry
        endpoint = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        _set_overpass_endpoint(endpoint)
        try:
            if attempt > 0:
                wait = base_wait * (2 ** (attempt - 1)) + random.uniform(0, 5)
                print(f"    Retry {attempt}/{max_retries - 1} via {endpoint} — waiting {wait:.1f}s ...")
                time.sleep(wait)
            return fn(*args, **kwargs)
        except Exception as e:
            last_exc = e
            print(f"    Attempt {attempt + 1} failed: {e}")
    raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_exc}")


def download_poi(city: str, boundary: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Download POIs with retry + endpoint rotation."""
    polygon = boundary.geometry.union_all()
    def _fetch():
        return ox.features_from_polygon(polygon,  tags={k: v for d in POI_CATEGORIES.values() for k, v in d.items()})
    gdf = _with_retry(_fetch)
    # ... rest of your existing POI processing logic
    return gdf


def download_road_network(city: str, boundary: gpd.GeoDataFrame):
    """Download road network with retry + endpoint rotation."""
    polygon = boundary.geometry.union_all()
    def _fetch():
        G = ox.graph_from_polygon(polygon, network_type="drive")
        return ox.graph_to_gdfs(G)
    nodes, edges = _with_retry(_fetch)
    # ... rest of your existing road processing logic
    return nodes, edges


def process_city(city: str) -> dict:
    print(f"\n{'='*65}")
    print(f"  {city.upper()}")
    print(f"{'='*65}")

    boundary = load_boundary(city)
    print(f"  Boundary loaded: {len(boundary)} feature(s)")
    summary = {"city": city}

    # ── POI ───────────────────────────────────────────────────────────────
    poi_path = poi_output_path(city)
    if poi_path.exists():
        existing = gpd.read_file(poi_path)
        print(f"  POI         : ⏭️  already on disk ({len(existing):,} features) — skipping")
        summary["poi_features"] = len(existing)
        summary["poi_path"]     = str(poi_path)
    else:
        print(f"  POI         : downloading...")
        try:
            poi_gdf = download_poi(city, boundary)
            if not poi_gdf.empty:
                poi_gdf.to_file(poi_path, driver="GPKG")
                print(f"  POI         : ✅ {len(poi_gdf):,} features saved")
            else:
                print(f"  POI         : ⚠️  empty — no file written")
            summary["poi_features"] = len(poi_gdf)
            summary["poi_path"]     = str(poi_path) if not poi_gdf.empty else None
        except Exception as e:
            print(f"  POI         : ❌ failed after all retries — {e}")
            summary["poi_features"] = None
            summary["poi_path"]     = None

    # ── Road network ──────────────────────────────────────────────────────
    edges_path = edges_output_path(city)
    nodes_path = nodes_output_path(city)
    if edges_path.exists() and nodes_path.exists():
        n_edges = len(gpd.read_file(edges_path))
        n_nodes = len(gpd.read_file(nodes_path))
        print(f"  Road network: ⏭️  already on disk ({n_edges:,} edges, {n_nodes:,} nodes) — skipping")
        summary["road_edges"] = n_edges
        summary["road_nodes"] = n_nodes
    else:
        print(f"  Road network: downloading...")
        try:
            nodes, edges = download_road_network(city, boundary)
            edges.to_file(edges_path, driver="GPKG")
            nodes.to_file(nodes_path, driver="GPKG")
            print(f"  Road network: ✅ {len(edges):,} edges, {len(nodes):,} nodes saved")
            summary["road_edges"] = len(edges)
            summary["road_nodes"] = len(nodes)
        except Exception as e:
            print(f"  Road network: ❌ failed after all retries — {e}")
            summary["road_edges"] = None
            summary["road_nodes"] = None

    # ── Inter-city pause to avoid hammering OSM ───────────────────────────
    time.sleep(random.uniform(5, 10))
    return summary

## 4. Pre-flight Check

Verifies that all clipped boundary files exist before downloading anything.
Run notebook 00a for any city flagged as missing.

In [ ]:
print("Checking clipped boundary files...")
missing = []

for city in CITIES:
    path = clipped_boundary_path(city)
    if path.exists():
        print(f"  ✅  {city}")
    else:
        print(f"  ❌  {city} — missing: {path}")
        missing.append(city)

if missing:
    raise FileNotFoundError(
        f"\nClipped boundary files missing for: {missing}\n"
        f"Run notebook 00a for these cities before proceeding."
    )

print(f"\nAll {len(CITIES)} boundary files found. Ready to download.")

## 5. Run Downloads

Processes each city sequentially. For each city:
- POI and road network are skipped if the output files already exist on disk.
- OSMnx caches OSM responses so re-running after an interruption is fast.
- A summary table is printed at the end.

In [ ]:
results = []
for city in CITIES:
    try:
        summary = process_city(city)
        summary["status"] = "✅ ok"
    except Exception as e:
        print(f"\n  ⚠️  {city} failed: {e}")
        summary = {"city": city, "status": f"⚠️  {e}"}
    results.append(summary)

print(f"\n{'='*65}")
print("  DOWNLOAD SUMMARY")
print(f"{'='*65}")
summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))

## 6. Output File Check

Lists all files written per city with their sizes.

In [ ]:
print("Files written per city:\n")
for city in CITIES:
    dirs = [
        RAW_DIR / city / "poi",
        RAW_DIR / city / "road_network",
    ]
    files = []
    for d in dirs:
        if d.exists():
            files.extend(sorted(d.glob("*.gpkg")))

    if not files:
        print(f"  {city}: no files found")
        continue

    print(f"  {city}:")
    for f in files:
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"    {str(f.relative_to(PROJECT_ROOT)):<70}  {size_mb:>6.2f} MB")